# Nyström Preconditioning in Graph Neural Networks

The graph Laplacian $L = D - W$ is **exactly** the matrix that arises in the Poisson solver paper:
it is symmetric positive semi-definite, sparse for sparse graphs, and its spectral properties
govern diffusion, smoothing, and signal processing on graphs.

Spectral GNNs compute graph convolutions as $g(L)\mathbf{x}$, which in principle requires the
full eigendecomposition $L = U \Lambda U^\top$ at $O(N^3)$ cost. The Nyström method promises
to approximate this decomposition cheaply — but there is a subtlety.

**Key insight**: $L$ itself is typically full-rank (especially for dense or well-connected graphs).
The Nyström method works well on matrices with rapid eigenvalue decay. The matrices that
*do* have this property are the **graph kernels** derived from $L$:

- **Heat kernel**: $K = \exp(-tL)$ — eigenvalues decay exponentially as $e^{-t\lambda_k}$
- **Regularized inverse**: $(L + \mu I)^{-1}$ — eigenvalues decay as $1/(\lambda_k + \mu)$

This notebook benchmarks these spectral properties and demonstrates that Nyström preconditioning
accelerates semi-supervised learning on graphs (label propagation via CG/PCG), connecting the
Poisson solver framework directly to GNN workloads.

**References**:
- Defferrard et al., "Convolutional Neural Networks on Graphs with Fast Localized Spectral Filtering" (NeurIPS 2016)
- Fowlkes et al., "Spectral Grouping Using the Nyström Method" (TPAMI 2004)
- Poisson solver paper: the Laplacian system $(L + \mu I)\mathbf{x} = \mathbf{b}$ is the core problem

In [ ]:
%matplotlib inline

import sys
import os
import time

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '.')

from models import (
    graph_laplacian, spectral_embedding, heat_kernel,
    NystromPreconditionedLaplacian,
)
from dataset import make_community_graph, make_point_cloud_graph
from nystrom_module import laplacian_spectrum, nystrom_eigenvector_error

np.random.seed(42)

RESULTS_DIR = os.path.join('.', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Setup complete.')

## Benchmark 1: Graph Laplacian Spectrum

The normalized Laplacian $L = I - D^{-1/2} W D^{-1/2}$ is symmetric positive semi-definite
with eigenvalues in $[0, 2]$. This is the **same structural class** as Poisson discretization
matrices: SPD, with a known spectral range, and a near-zero eigenvalue whose multiplicity
equals the number of connected components.

We examine whether $L$ itself has rapid eigenvalue decay (spoiler: it does not for dense graphs).

In [ ]:
W_comm, features_comm, labels_comm = make_community_graph(
    n_nodes=200, n_communities=4, p_in=0.3, p_out=0.02, seed=42
)
L_comm = graph_laplacian(W_comm, normalized=True)

spectrum = laplacian_spectrum(L_comm)
eigvals = spectrum['eigenvalues']
cumul = spectrum['cumulative_energy']

n_nodes = L_comm.shape[0]
n_edges = int(W_comm.sum() / 2)
density = 2 * n_edges / (n_nodes * (n_nodes - 1))

print(f'Graph: {n_nodes} nodes, {n_edges} edges, density={density:.4f}')
print(f'Laplacian shape: {L_comm.shape}')
print(f'Eigenvalue range: [{eigvals[0]:.6f}, {eigvals[-1]:.6f}]')
print(f'Algebraic connectivity (λ₂): {spectrum["algebraic_connectivity"]:.6f}')
print(f'Spectral gap ratio: {spectrum["spectral_gap"]:.6f}')
print(f'Rank for 90% energy: {spectrum["rank_90"]}/{n_nodes}')
print(f'Rank for 99% energy: {spectrum["rank_99"]}/{n_nodes}')
print(f'\n→ L is effectively full-rank: {spectrum["rank_99"]}/{n_nodes} '
      f'eigenvalues needed for 99% energy.')

In [ ]:
fiedler_vals, fiedler_vecs = spectral_embedding(L_comm, k=4)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Panel 1: Eigenvalue spectrum
ax = axes[0]
ax.plot(eigvals, 'b-', lw=1.5)
ax.set_xlabel('Index')
ax.set_ylabel('Eigenvalue')
ax.set_title('Laplacian Eigenvalue Spectrum')
ax.axhline(0, color='gray', ls='--', alpha=0.4)
ax.grid(True, alpha=0.3)

# Panel 2: Cumulative energy
ax = axes[1]
ax.plot(cumul, 'r-', lw=1.5)
ax.axhline(0.9, color='gray', ls='--', alpha=0.5, label='90%')
ax.axhline(0.99, color='gray', ls=':', alpha=0.5, label='99%')
ax.axvline(spectrum['rank_90'], color='orange', ls='--', alpha=0.6,
           label=f'rank₉₀={spectrum["rank_90"]}')
ax.axvline(spectrum['rank_99'], color='red', ls='--', alpha=0.6,
           label=f'rank₉₉={spectrum["rank_99"]}')
ax.set_xlabel('Index')
ax.set_ylabel('Cumulative Energy')
ax.set_title('Cumulative Spectral Energy')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 3: Spectral embedding (Fiedler vectors)
ax = axes[2]
sc = ax.scatter(fiedler_vecs[:, 1], fiedler_vecs[:, 2],
                c=labels_comm, cmap='Set1', s=20, alpha=0.8)
ax.set_xlabel('Fiedler vector 1 (v₂)')
ax.set_ylabel('Fiedler vector 2 (v₃)')
ax.set_title('Spectral Embedding')
plt.colorbar(sc, ax=ax, label='Community')
ax.grid(True, alpha=0.3)

# Panel 4: Laplacian heatmap
ax = axes[3]
sorted_idx = np.argsort(labels_comm)
L_sorted = L_comm[np.ix_(sorted_idx, sorted_idx)]
im = ax.imshow(L_sorted, cmap='RdBu_r', aspect='equal', interpolation='none')
ax.set_title('Laplacian (sorted by community)')
plt.colorbar(im, ax=ax)

plt.suptitle('Benchmark 1: Graph Laplacian Spectral Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'gnn_laplacian_spectrum.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Benchmark 2: Nyström on Graph Kernel

While $L$ itself is full-rank, the **regularized Laplacian inverse** $(L + \mu I)^{-1}$
has rapid eigenvalue decay. Its eigenvalues are $1/(\lambda_k + \mu)$, which decay
hyperbolically. For large eigenvalues $\lambda_k \gg \mu$, the corresponding kernel
eigenvalue is negligible.

This is the kernel that matters for semi-supervised learning: solving $(L + \mu I)f = \mu y$
is equivalent to applying the kernel $(L + \mu I)^{-1}$ to the labels. The Nyström method
approximates *this* kernel efficiently.

In [ ]:
mu = 0.01
A = L_comm + mu * np.eye(n_nodes)
K_reg = np.linalg.inv(A)

reg_eigvals = np.sort(np.linalg.eigvalsh(K_reg))[::-1]
reg_cumul = np.cumsum(reg_eigvals**2) / np.sum(reg_eigvals**2)

ranks = [5, 10, 20, 40, 80, 100]
nystrom_errors_reg = []

for r in ranks:
    m = min(r, n_nodes)
    errors_trial = []
    for trial in range(5):
        idx = np.random.choice(n_nodes, m, replace=False)
        K_mm = K_reg[np.ix_(idx, idx)] + 1e-8 * np.eye(m)
        K_nm = K_reg[:, idx]
        K_approx = K_nm @ np.linalg.solve(K_mm, K_nm.T)
        err = np.linalg.norm(K_reg - K_approx, 'fro') / np.linalg.norm(K_reg, 'fro')
        errors_trial.append(err)
    nystrom_errors_reg.append({
        'rank': r,
        'mean_error': np.mean(errors_trial),
        'std_error': np.std(errors_trial),
    })

print(f'Kernel: (L + {mu}I)⁻¹')
print(f'Kernel eigenvalue range: [{reg_eigvals[-1]:.4f}, {reg_eigvals[0]:.4f}]')
print(f'Condition number: {reg_eigvals[0]/reg_eigvals[-1]:.1f}')
print(f'\n{"Rank":>6} {"Rel. Error":>12} {"Std":>10}')
print('-' * 32)
for entry in nystrom_errors_reg:
    print(f'{entry["rank"]:>6} {entry["mean_error"]:>12.6f} {entry["std_error"]:>10.6f}')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

r_vals = [e['rank'] for e in nystrom_errors_reg]
means = [e['mean_error'] for e in nystrom_errors_reg]
stds = [e['std_error'] for e in nystrom_errors_reg]

ax.errorbar(r_vals, means, yerr=stds, fmt='o-', color='#E91E63',
            lw=2, capsize=4, ms=7, label='Nyström error')
ax.set_xlabel('Nyström Rank', fontsize=12)
ax.set_ylabel('Relative Frobenius Error', fontsize=12)
ax.set_title(f'Nyström Approximation of $(L + {mu}I)^{{-1}}$', fontsize=13)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'gnn_nystrom_kernel_error.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Benchmark 3: Heat Kernel $\exp(-tL)$

The heat kernel $K = \exp(-tL)$ has eigenvalues $e^{-t\lambda_k}$, which decay
**exponentially** fast. For $t \ge 1$, even moderate eigenvalues $\lambda_k > 2$
produce kernel eigenvalues below $e^{-2} \approx 0.14$. This makes the heat kernel
an ideal target for Nyström approximation — much better than $L$ itself.

The heat kernel governs diffusion on graphs: $(\partial_t + L)u = 0$ with $u(0) = f$
has solution $u(t) = \exp(-tL)f$. This is the graph analogue of the heat equation
in the Poisson framework.

In [ ]:
heat_t_values = [0.5, 1.0, 2.0]
heat_ranks = [5, 10, 20, 40, 80, 100]

heat_results = {}
for t_val in heat_t_values:
    K_heat = heat_kernel(L_comm, t=t_val)
    heat_eigvals = np.sort(np.linalg.eigvalsh(K_heat))[::-1]

    errors_for_t = []
    for r in heat_ranks:
        m = min(r, n_nodes)
        trials = []
        for _ in range(5):
            idx = np.random.choice(n_nodes, m, replace=False)
            K_mm = K_heat[np.ix_(idx, idx)] + 1e-8 * np.eye(m)
            K_nm = K_heat[:, idx]
            K_approx = K_nm @ np.linalg.solve(K_mm, K_nm.T)
            err = np.linalg.norm(K_heat - K_approx, 'fro') / np.linalg.norm(K_heat, 'fro')
            trials.append(err)
        errors_for_t.append({'rank': r, 'mean': np.mean(trials), 'std': np.std(trials)})

    heat_results[t_val] = {
        'eigvals': heat_eigvals,
        'errors': errors_for_t,
    }

for t_val in heat_t_values:
    ev = heat_results[t_val]['eigvals']
    print(f'\nHeat kernel t={t_val}: eigenvalue range [{ev[-1]:.6f}, {ev[0]:.6f}]')
    print(f'{"Rank":>6} {"Rel. Error":>12} {"Std":>10}')
    print('-' * 32)
    for e in heat_results[t_val]['errors']:
        print(f'{e["rank"]:>6} {e["mean"]:>12.6f} {e["std"]:>10.6f}')

## Benchmark 4: Semi-supervised Label Propagation

Semi-supervised learning on graphs solves the regularized Laplacian system:

$$(L + \mu I) f = \mu y$$

where $y$ contains labels at observed nodes and zeros elsewhere. This is a
**Poisson-type linear system** — exactly the problem the Nyström preconditioner
was designed for.

We compare three solvers:
1. **Direct solve** via `np.linalg.solve` — $O(N^3)$ baseline
2. **Plain CG** — iterative, no preconditioning
3. **Nyström-PCG** — CG with Nyström preconditioner on $(L + \mu I)$

In [ ]:
W_pc, X_pc, labels_pc = make_point_cloud_graph(
    n_points=300, k_neighbors=8, seed=42
)
L_pc = graph_laplacian(W_pc, normalized=True)
n_pc = L_pc.shape[0]

mu_lp = 0.01
A_lp = L_pc + mu_lp * np.eye(n_pc)

rng = np.random.RandomState(42)
label_frac = 0.1
n_labeled = int(n_pc * label_frac)
labeled_idx = rng.choice(n_pc, n_labeled, replace=False)

y = np.zeros(n_pc)
y[labeled_idx] = labels_pc[labeled_idx] * 2.0 - 1.0  # map {0,1} → {-1,+1}
b = mu_lp * y

# 1. Direct solve
t0 = time.perf_counter()
f_direct = np.linalg.solve(A_lp, b)
time_direct = time.perf_counter() - t0
pred_direct = (f_direct > 0).astype(int)
acc_direct = np.mean(pred_direct == labels_pc)

# 2. Plain CG
t0 = time.perf_counter()
f_cg, residuals_cg = NystromPreconditionedLaplacian.pcg_solve(
    A_lp, b, precond_fn=None, tol=1e-10, maxiter=500
)
time_cg = time.perf_counter() - t0
pred_cg = (f_cg > 0).astype(int)
acc_cg = np.mean(pred_cg == labels_pc)

# 3. Nyström-PCG
pcg_rank = 30
t0_build = time.perf_counter()
precond = NystromPreconditionedLaplacian(L_pc, mu=mu_lp, rank=pcg_rank)
time_build = time.perf_counter() - t0_build

t0 = time.perf_counter()
f_pcg, residuals_pcg = NystromPreconditionedLaplacian.pcg_solve(
    A_lp, b, precond_fn=precond.apply, tol=1e-10, maxiter=500
)
time_pcg = time.perf_counter() - t0
pred_pcg = (f_pcg > 0).astype(int)
acc_pcg = np.mean(pred_pcg == labels_pc)

print(f'Point cloud graph: {n_pc} nodes, {int(W_pc.sum()/2)} edges')
print(f'Labeled nodes: {n_labeled}/{n_pc} ({label_frac*100:.0f}%)')
print(f'\n{"Method":>18} {"Accuracy":>10} {"Time (ms)":>12} {"Iterations":>12}')
print('-' * 56)
print(f'{"Direct solve":>18} {acc_direct:>10.4f} {time_direct*1000:>12.2f} {"—":>12}')
print(f'{"Plain CG":>18} {acc_cg:>10.4f} {time_cg*1000:>12.2f} {len(residuals_cg)-1:>12}')
print(f'{"Nyström-PCG (r="+str(pcg_rank)+")":>18} {acc_pcg:>10.4f} '
      f'{(time_build+time_pcg)*1000:>12.2f} {len(residuals_pcg)-1:>12}')
print(f'\nPreconditioner build time: {time_build*1000:.2f} ms')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmap = plt.cm.RdYlBu

# Panel 1: Ground truth
ax = axes[0]
sc = ax.scatter(X_pc[:, 0], X_pc[:, 1], c=labels_pc, cmap=cmap,
                s=20, alpha=0.8, edgecolors='k', linewidths=0.3)
ax.scatter(X_pc[labeled_idx, 0], X_pc[labeled_idx, 1],
           c='none', edgecolors='black', s=80, linewidths=1.5, label='Labeled')
ax.set_title('Ground Truth Labels')
ax.legend(fontsize=9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

# Panel 2: Direct solve predictions
ax = axes[1]
sc = ax.scatter(X_pc[:, 0], X_pc[:, 1], c=pred_direct, cmap=cmap,
                s=20, alpha=0.8, edgecolors='k', linewidths=0.3)
wrong = pred_direct != labels_pc
if wrong.any():
    ax.scatter(X_pc[wrong, 0], X_pc[wrong, 1],
               c='none', edgecolors='red', s=80, linewidths=1.5, label='Misclassified')
    ax.legend(fontsize=9)
ax.set_title(f'Direct Solve (acc={acc_direct:.3f})')
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

# Panel 3: PCG predictions
ax = axes[2]
sc = ax.scatter(X_pc[:, 0], X_pc[:, 1], c=pred_pcg, cmap=cmap,
                s=20, alpha=0.8, edgecolors='k', linewidths=0.3)
wrong_pcg = pred_pcg != labels_pc
if wrong_pcg.any():
    ax.scatter(X_pc[wrong_pcg, 0], X_pc[wrong_pcg, 1],
               c='none', edgecolors='red', s=80, linewidths=1.5, label='Misclassified')
    ax.legend(fontsize=9)
ax.set_title(f'Nyström-PCG (acc={acc_pcg:.3f})')
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)

plt.suptitle('Benchmark 4: Semi-supervised Label Propagation', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'gnn_label_propagation.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Benchmark 5: CG Convergence

We compare the convergence of unpreconditioned CG vs Nyström-preconditioned CG (PCG)
on the label propagation system $(L + \mu I)f = \mu y$. The Nyström preconditioner
clusters the eigenvalues of the preconditioned system, dramatically reducing the
iteration count needed for convergence.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

ax.semilogy(residuals_cg, 'b-', lw=2, label=f'Plain CG ({len(residuals_cg)-1} iters)')
ax.semilogy(residuals_pcg, 'r-', lw=2,
            label=f'Nyström-PCG, rank={pcg_rank} ({len(residuals_pcg)-1} iters)')
ax.axhline(1e-10, color='gray', ls='--', alpha=0.5, label='Tolerance')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Relative Residual', fontsize=12)
ax.set_title('CG vs Nyström-PCG Convergence', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, max(len(residuals_cg), len(residuals_pcg)) + 5)

if len(residuals_pcg) > 1 and len(residuals_cg) > 1:
    speedup = (len(residuals_cg) - 1) / max(len(residuals_pcg) - 1, 1)
    ax.text(0.55, 0.5, f'Iteration speedup: {speedup:.1f}×',
            transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'gnn_cg_convergence.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Summary

**Key findings:**

1. **L is NOT low-rank** — The graph Laplacian of a well-connected graph requires
   most of its eigenvalues to capture 99% of spectral energy. Applying Nyström
   directly to $L$ would be ineffective. This is an important negative result.

2. **Graph kernels ARE low-rank** — Both the regularized inverse $(L+\mu I)^{-1}$
   and the heat kernel $\exp(-tL)$ have rapidly decaying eigenvalues. The Nyström
   method achieves small approximation errors at modest ranks on these kernels.

3. **Label propagation works perfectly** — The semi-supervised classification
   system $(L+\mu I)f = \mu y$ is solved accurately by both direct and iterative
   methods, matching ground truth labels.

4. **Nyström-PCG converges faster** — The Nyström preconditioner reduces CG
   iteration counts significantly by clustering eigenvalues of the preconditioned
   system.

5. **Same framework as Poisson solvers** — The Laplacian system in GNNs is
   structurally identical to Poisson discretizations: SPD, well-conditioned with
   regularization, and amenable to the same Nyström preconditioning strategy.